In [ ]:
!pip install -q langchain langchain-openai langchain_mongodb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.6/50.6 kB 424.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 407.7/407.7 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.9/296.9 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.9/386.9 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.6/313.6 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.0/78.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.2/325.2 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.

In [ ]:
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from google.colab import userdata

llm = AzureChatOpenAI(
    model=userdata.get('AZURE_MODEL_NAME'),
    deployment_name=userdata.get('AZURE_MODEL_NAME'),
    temperature=0,
    # max_tokens=None,
    timeout=None,
    max_retries=2,
    api_key=userdata.get('AZURE_API_KEY'),  # if you prefer to pass api key in directly instaed of using env vars
    azure_endpoint=userdata.get('AZURE_BASE_URL'),
    api_version=userdata.get('AZURE_API_VERSION'),
)
#
embd = AzureOpenAIEmbeddings(
    model=userdata.get('AZURE_EMBEDDING_NAME'),
    api_key=userdata.get('AZURE_API_KEY'),
    azure_endpoint=userdata.get('AZURE_BASE_URL'),
    api_version=userdata.get('AZURE_API_VERSION'),
)

In [ ]:
from langchain_mongodb.vectorstores import MongoDBAtlasVectorSearch
from google.colab import userdata
from pymongo import MongoClient

# initialize MongoDB python client
client = MongoClient(userdata.get('DELOITTE_MONGODB_URI'))
DB_NAME = "CWS"
COLLECTION_NAME = "cv_data"
ATLAS_VECTOR_SEARCH_INDEX_NAME = "candidateSemanticSearch"

MONGODB_COLLECTION = client[DB_NAME][COLLECTION_NAME]

vector_store = MongoDBAtlasVectorSearch(
    collection=MONGODB_COLLECTION,
    embedding=embd,
    index_name=ATLAS_VECTOR_SEARCH_INDEX_NAME,
    embedding_key="embeddings",
    relevance_score_fn="cosine",#"MMR",#"cosine",
    text_key="filename" # ID of document was extracted
)

In [ ]:
results = vector_store.max_marginal_relevance_search(
    "Who people has education in Maryland Global Campus? list all candidate",
    k=2, fetch_k=50,lambda_mult=0.9
)
# results
[(res.page_content, res.metadata['summary']) for res in results]

[('Mozelle Abercrombie_email.pdf',
  'Mozelle Abercrombie has over 6 years of experience in Java and Object-Oriented Analysis & Design, with expertise in web-based and legacy applications. Skilled in various Java IDEs and application servers, Mozelle has a strong background in product development, distributed applications, and database management. Known for excellent communication and problem-solving skills.'),
 ('Celsa  Borland_email.pdf',
  'Alicia is a FOIA Analyst with over 18 years of experience. She has served as an independent FOIA case action officer for cases with the Office of Secretary of Defense, Joint Services, or Congressional components. She has prepared detailed written report responses or a Memorandum for Record (MFR) Redact responses per FOIA exemption guidelines. She has exercised discretionary judgment of FOIA procedural issues and evaluated FOIA responses by evaluating facts and performing the appropriate research. She has also supported FOIA Requestor Service Cent

In [ ]:
results = vector_store.similarity_search_with_score(
    "Who people has education in Maryland Global Campus? list all candidate", k=4, # pre_filter=...,
)
# results
[(score, res.page_content, res.metadata['additional_info']) for res, score in results]

[(0.6611223220825195,
  'Mozelle Abercrombie_email.pdf',
  'Excellent communication, analytical and interpersonal skills were demonstrated. Exceptional ability to learn and master new technologies and to deliver outputs in short deadlines was shown. Hands-on problem solving and development, results oriented, team lead and team member experience, effective communicator at all levels.'),
 (0.6511450409889221,
  'Celsa  Borland_email.pdf',
  'Job-related training was completed.'),
 (0.6494563817977905,
  'Borland Alicia_email.pdf',
  'Job-related training was completed.'),
 (0.643711507320404,
  'Kent Abercrombie 2.pdf',
  'Enjoyed empowering people and building team rapport. Strongly committed to quality process.')]

In [ ]:
retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 1, "score_threshold": 0.2},
)
retriever.invoke("Who people has education in Maryland Global Campus? list all candidate")

[Document(metadata={'_id': '66e9c4133c821c1e1d27e249', 'security_id': 'TESTWen20489-WM817EPm10UI', 'num_pages': 7, 'data': "01/30/2024\nMozelle Abercrombie\nSummary\n· Over 6 Years of experience in Java (J2ME, J2EE, J2SE) and Object Oriented Analysis & Design\n(OOA&D), Development and Implementation of Web based and Legacy applications\n· Experience in designing class diagrams, sequence diagrams using UML, Rational Rose and RUP\n(Rational Unified Process)\n· Areas of expertise include STRUTS (MVC), J2EE Components developed and deployed in Java\nBean, EJB, Graphical User Interface (GUI) design using Java Swings and AWT\n· Hands on experience on Application Servers like IBM WebSphere, BEA WebLogic, iPlanet 6.5 (SUN\nOne) and Java, Apache 1.3 & IIS web servers\n· Skilled at programming on different Java IDE’ s like Borland JBuilder, WebGain Visual Café, Junit, IBM\nWSAD and MS Visual InterDev, MS Front Page for ASP\n· Expertise in the areas of Product development, Distributed application

In [ ]:
from langchain import hub
from langchain.chains import RetrievalQA

# See full prompt at https://smith.langchain.com/hub/rlm/rag-prompt
prompt = hub.pull("rlm/rag-prompt")

qa_chain = RetrievalQA.from_llm(
    llm, retriever=vector_store.as_retriever(), prompt=prompt
)

/usr/local/lib/python3.10/dist-packages/langsmith/client.py:354: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [ ]:
qa_chain.invoke("Who people has education in Maryland Global Campus? list all candidates")

{'query': 'Who people has education in Maryland Global Campus? list all candidates',
 'result': 'Mozelle Abercrombie and Celsa Borland have education in Maryland Global Campus.'}

In [ ]:
from langchain import hub
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# See full prompt at https://smith.langchain.com/hub/rlm/rag-prompt
prompt = hub.pull("rlm/rag-prompt")


def format_docs(docs):
    return "\n\n".join(str(doc.metadata) for doc in docs)
qa_chain = (
    {
        "context": vector_store.as_retriever() | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

/usr/local/lib/python3.10/dist-packages/langsmith/client.py:354: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [ ]:
qa_chain.invoke("Who has hard skill in php?")

'Amitendra Gaurav has a hard skill in PHP.'

In [ ]:
# %%timeit -n3 -r1
print(qa_chain.invoke("Tell me about name Amitendra Gaurav, their skills and qualifications, using bullet points"))

- **Name**: Amitendra Gaurav
- **Email**: gamitendra@yahoo.com
- **Skills**:
  - Core Java, HTML, CSS, JavaScript, PHP, Basic C
  - Microsoft Office (MS Word, PowerPoint, Excel)
  - Internet proficiency
  - Operating Systems (Windows XP, Windows 7, Windows 8, RedHat Linux 7)
  - CCNA
- **Qualifications**:
  - B.E from RGTU, 2014
  - 12th from M.P. BSE, 2009
  - 10th from M.P. BSE, 2007
- **Training**:
  - 45 days CCNA training from Krnetwork Delhi
  - 2 months C/C++ training from DUCAT Gwalior
  - 3 months RedHat Linux 7 training from Krnetwork Delhi


In [ ]:
from langchain import hub
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# See full prompt at https://smith.langchain.com/hub/rlm/rag-prompt
prompt = hub.pull("rlm/rag-prompt")


def format_docs(docs):
    return "\n\n".join(str(doc.metadata) for doc in docs)
qa_chain = (
    {
        "context": vector_store.as_retriever(search_kwargs={"k": 30}) | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

/usr/local/lib/python3.10/dist-packages/langsmith/client.py:354: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [ ]:
print(qa_chain.invoke("""
Which candidates has similar information to Ardra Prasad, their skills and qualifications,
using a matching markdown table(4 columns), then tell me why in bullet points.
Finally generate insights.
"""))

| Name                | Degree/Qualification                  | Skills/Experience                                                                 | Email                        |
|---------------------|---------------------------------------|----------------------------------------------------------------------------------|------------------------------|
| Ayush Pandey        | B.Tech in Mechanical Engineering      | AutoCAD, ProCAST, UG-NX, Solid-Works, ANSYS, SolidCAM, Reverse Engineering        | shub.pandey7@yahoo.in        |
| Amitendra Gaurav    | B.E in Engineering                    | HTML, CSS, JavaScript, PHP, Microsoft Office, Internet, Operating Systems         | gamitendra@yahoo.com         |
| Mozelle Abercrombie | Masters in Computer Applications (MCA)| Java, J2EE, JDBC, JSP, EJB, Servlets, Struts, SQL, PL/SQL, Web Technologies       | aber.moz@outlook.com         |
| Kasandra Pennant    | Masters in Computer and Information Systems | Java, HTML, CSS, Angular, Spring, AW

In [ ]:
qa_retriever = vector_store.as_retriever(
   search_type="similarity",
   search_kwargs={
       "k": 200,
       "post_filter_pipeline": [{"$limit": 25}]
   }
)